In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/satellite_anomaly_detection")

for folder in ["data", "models", "results"]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project setup complete")
print("Python version:", sys.version.split()[0])
print("Project folder:", PROJECT_DIR)

Project setup complete
Python version: 3.13.15
Project folder: /content/satellite_anomaly_detection


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

SAVE_DIR = Path(
    "/content/drive/MyDrive/Satellite_Anomaly_Detection_v1"
)

for folder in ["data", "models", "results"]:
    (SAVE_DIR / folder).mkdir(parents=True, exist_ok=True)

# Check that saving works
check_file = SAVE_DIR / "setup_check.txt"
check_file.write_text("Project saving setup successful.\n")

assert check_file.read_text() == "Project saving setup successful.\n"

print("Drive saving verified!")
print("Save location:", SAVE_DIR)

Mounted at /content/drive
Drive saving verified!
Save location: /content/drive/MyDrive/Satellite_Anomaly_Detection_v1


In [ ]:
import sys
import json
import importlib

versions = {"python": sys.version.split()[0]}

for name in ["numpy", "pandas", "sklearn", "tensorflow"]:
    try:
        module = importlib.import_module(name)
        versions[name] = module.__version__
        print(f"{name}: {module.__version__}")
    except Exception as error:
        versions[name] = f"ERROR: {error}"
        print(f"{name}: ERROR — {error}")

environment_file = SAVE_DIR / "results" / "environment.json"
environment_file.write_text(json.dumps(versions, indent=2))

print("\nEnvironment details saved.")

numpy: 2.1.3
pandas: 2.2.3
sklearn: 1.6.1
tensorflow: 2.20.0

Environment details saved.


In [ ]:
import hashlib
import urllib.request
import pandas as pd

DATA_DIR = SAVE_DIR / "data" / "opssat_ad_v2"
DATA_DIR.mkdir(parents=True, exist_ok=True)

base_url = "https://zenodo.org/records/15108715/files"

files = {
    "segments.csv": "72f109630abb933a386106897a631188",
    "dataset.csv": "5246fdc5e4630a4cecbf7fb6bc8b795e",
}

for filename, expected_md5 in files.items():
    path = DATA_DIR / filename

    if not path.exists():
        print("Downloading:", filename, flush=True)
        request = urllib.request.Request(
            f"{base_url}/{filename}?download=1",
            headers={"User-Agent": "SatelliteResearch/1.0"},
        )
        with urllib.request.urlopen(request, timeout=120) as response:
            content = response.read()

        if hashlib.md5(content).hexdigest() != expected_md5:
            raise ValueError(f"Download verification failed: {filename}")

        path.write_bytes(content)

    actual_md5 = hashlib.md5(path.read_bytes()).hexdigest()
    if actual_md5 != expected_md5:
        raise ValueError(f"Existing file checksum mismatch: {filename}")

    print(f"Verified: {filename} ({path.stat().st_size:,} bytes)")

# Inspect the format before choosing preprocessing
for filename in files:
    preview = pd.read_csv(DATA_DIR / filename, nrows=5)
    print(f"\n--- {filename} ---")
    print("Columns:", preview.columns.tolist())
    print(preview.head(3).to_string(index=False))

print("\nDataset download and verification complete.")

Downloading: segments.csv
Verified: segments.csv (18,987,091 bytes)
Downloading: dataset.csv
Verified: dataset.csv (507,550 bytes)

--- segments.csv ---
Columns: ['channel', 'timestamp', 'value', 'label', 'sampling', 'anomaly', 'segment', 'train']
 channel                timestamp     value   label  sampling  anomaly  segment  train
CADC0872 2022-06-01T23:42:54.000Z -0.000021 anomaly         1        1        1      1
CADC0872 2022-06-01T23:42:55.000Z -0.000021 anomaly         1        1        1      1
CADC0872 2022-06-01T23:42:56.000Z -0.000021 anomaly         1        1        1      1

--- dataset.csv ---
Columns: ['segment', 'anomaly', 'train', 'channel', 'sampling', 'duration', 'len', 'mean', 'var', 'std', 'kurtosis', 'skew', 'n_peaks', 'smooth10_n_peaks', 'smooth20_n_peaks', 'diff_peaks', 'diff2_peaks', 'diff_var', 'diff2_var', 'gaps_squared', 'len_weighted', 'var_div_duration', 'var_div_len']
 segment  anomaly  train  channel  sampling  duration  len          mean          var 

In [ ]:
import pandas as pd

segments = pd.read_csv(DATA_DIR / "segments.csv")
metadata = pd.read_csv(DATA_DIR / "dataset.csv")

print("Telemetry rows:", len(segments))
print("Metadata rows:", len(metadata))
print("Channels:", segments["channel"].nunique())

print("\nLabel mapping:")
print(
    segments[["label", "anomaly"]]
    .drop_duplicates()
    .to_string(index=False)
)

print("\nSplit values:")
print(metadata["train"].value_counts(dropna=False).to_string())

print("\nSegments by channel, split and anomaly label:")
summary = (
    metadata.groupby(["channel", "train", "anomaly"], dropna=False)
    .size()
    .reset_index(name="segment_count")
)
print(summary.to_string(index=False))

print("\nSegment lengths:")
print(metadata["len"].describe().to_string())

print("\nMissing values in telemetry:")
print(segments.isna().sum().to_string())

summary.to_csv(
    SAVE_DIR / "results" / "dataset_summary.csv",
    index=False
)

print("\nDataset summary saved.")

Telemetry rows: 303493
Metadata rows: 2123
Channels: 9

Label mapping:
  label  anomaly
anomaly        1
anomaly        0

Split values:
train
1    1594
0     529

Segments by channel, split and anomaly label:
 channel  train  anomaly  segment_count
CADC0872      0        0            100
CADC0872      0        1             32
CADC0872      1        0            315
CADC0872      1        1             99
CADC0873      0        0            122
CADC0873      0        1             31
CADC0873      1        0            366
CADC0873      1        1             74
CADC0874      0        0             29
CADC0874      0        1             23
CADC0874      1        0             96
CADC0874      1        1             46
CADC0884      0        0             36
CADC0884      1        0            122
CADC0886      0        0              3
CADC0886      0        1              1
CADC0886      1        0              5
CADC0886      1        1              2
CADC0888      0        0      

In [ ]:
import numpy as np

keys = ["channel", "segment"]

# Each segment must have exactly one metadata record
assert not metadata.duplicated(keys).any(), \
    "Duplicate segment records in metadata"

# Each telemetry segment must have consistent labels and split
grouped = segments.groupby(keys)
consistency = grouped[["anomaly", "train"]].nunique()

assert consistency.eq(1).all().all(), \
    "Conflicting labels or split values within a segment"

# Compare telemetry records with metadata
observed = grouped.agg(
    actual_length=("value", "size"),
    observed_anomaly=("anomaly", "first"),
    observed_train=("train", "first"),
).reset_index()

checked = metadata.merge(
    observed,
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

assert checked["_merge"].eq("both").all(), \
    "Some segments are missing from one file"
assert checked["anomaly"].eq(checked["observed_anomaly"]).all(), \
    "Anomaly labels disagree between files"
assert checked["train"].eq(checked["observed_train"]).all(), \
    "Train/test assignments disagree between files"
assert checked["len"].eq(checked["actual_length"]).all(), \
    "Segment lengths disagree"
assert np.isfinite(segments["value"].to_numpy(dtype=float)).all(), \
    "Telemetry contains non-finite values"

checked.drop(columns="_merge").to_csv(
    SAVE_DIR / "results" / "segment_integrity_check.csv",
    index=False,
)

print("PASS: All segments match across both files.")
print("PASS: Numeric labels, split assignments and lengths agree.")
print("PASS: All telemetry values are finite.")

PASS: All segments match across both files.
PASS: Numeric labels, split assignments and lengths agree.
PASS: All telemetry values are finite.


In [ ]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

split_table = metadata[
    ["channel", "segment", "anomaly", "train"]
].copy()

assert split_table["train"].isin([0, 1]).all()
assert split_table["anomaly"].isin([0, 1]).all()

split_table["subset"] = "test"
development = split_table["train"].eq(1)
split_table.loc[development, "subset"] = "train"

# Hold out about 20% of each channel/class in the original training set.
# Split whole segments before creating any windows.
groups = split_table.loc[development].groupby(
    ["channel", "anomaly"], sort=True
)

for _, group in groups:
    indices = group.sort_values("segment").index.to_numpy()
    if len(indices) < 2:
        continue

    n_validation = min(
        len(indices) - 1,
        max(1, int(np.ceil(0.20 * len(indices))))
    )

    selected = rng.permutation(indices)[:n_validation]
    split_table.loc[selected, "subset"] = "validation"

# Anomalies remaining in training are excluded from AE fitting.
split_table["use_for_ae_fit"] = (
    split_table["subset"].eq("train")
    & split_table["anomaly"].eq(0)
)

assert not split_table.duplicated(["channel", "segment"]).any()
assert split_table.loc[
    split_table["train"].eq(0), "subset"
].eq("test").all()

split_table.to_csv(
    SAVE_DIR / "results" / "segment_split_v1.csv",
    index=False
)

print("Segment counts (anomaly: 0=normal, 1=anomalous):")
print(pd.crosstab(
    split_table["subset"], split_table["anomaly"]
).to_string())

print("\nNormal segments for AE training:",
      int(split_table["use_for_ae_fit"].sum()))
print("\nSplit saved: segment_split_v1.csv")

Segment counts (anomaly: 0=normal, 1=anomalous):
anomaly        0    1
subset               
test         416  113
train       1015  253
validation   258   68

Normal segments for AE training: 1015

Split saved: segment_split_v1.csv


In [ ]:
import json
import numpy as np

# Attach the saved split to each telemetry reading
prepared = segments.merge(
    split_table[
        ["channel", "segment", "subset", "use_for_ae_fit"]
    ],
    on=["channel", "segment"],
    how="left",
    validate="many_to_one",
)

assert prepared["subset"].notna().all()
assert len(prepared) == len(segments)

scaling = {}

for channel, group in prepared.groupby("channel"):
    training_values = group.loc[
        group["use_for_ae_fit"], "value"
    ].to_numpy(dtype=np.float64)

    assert len(training_values) > 0, \
        f"No normal training readings for {channel}"

    mean = float(training_values.mean())
    std = float(training_values.std(ddof=0))

    # Avoid division by zero for a constant training channel
    scale = std if std > 0 else 1.0
    scaling[channel] = {
        "mean": mean,
        "scale": scale,
        "constant_training_channel": bool(std == 0),
        "training_readings": int(len(training_values)),
    }

    prepared.loc[group.index, "scaled_value"] = (
        group["value"] - mean
    ) / scale

assert np.isfinite(prepared["scaled_value"]).all()

# Put readings in time order within each segment
prepared["timestamp"] = pd.to_datetime(
    prepared["timestamp"], utc=True, errors="raise"
)
prepared = prepared.sort_values(
    ["channel", "segment", "timestamp"]
).reset_index(drop=True)

(SAVE_DIR / "models" / "channel_scaling_v1.json").write_text(
    json.dumps(scaling, indent=2)
)

prepared.to_csv(
    DATA_DIR / "telemetry_scaled_v1.csv",
    index=False
)

print("Scaling complete — fitted on normal training data only.")
print("Channels scaled:", len(scaling))
print("Readings preserved:", len(prepared))
print("Constant training channels:",
      [c for c, s in scaling.items()
       if s["constant_training_channel"]])
print("Scaler and scaled telemetry saved.")

Scaling complete — fitted on normal training data only.
Channels scaled: 9
Readings preserved: 303493
Constant training channels: []
Scaler and scaled telemetry saved.


In [ ]:
import numpy as np
import pandas as pd
import json

WINDOW = 8
STRIDE = 4

windows = []
records = []

for (channel, segment), group in prepared.groupby(
    ["channel", "segment"], sort=True
):
    group = group.sort_values("timestamp")
    values = group["scaled_value"].to_numpy(dtype=np.float32)

    assert len(values) >= WINDOW, \
        f"Segment shorter than window: {channel}, {segment}"

    starts = list(range(0, len(values) - WINDOW + 1, STRIDE))

    # Include the final readings without padding
    last_start = len(values) - WINDOW
    if starts[-1] != last_start:
        starts.append(last_start)

    first = group.iloc[0]

    for start in starts:
        windows.append(values[start:start + WINDOW, None])
        records.append({
            "channel": channel,
            "segment": int(segment),
            "start_index": start,
            "subset": first["subset"],
            "segment_anomaly": int(first["anomaly"]),
            "use_for_ae_fit": bool(first["use_for_ae_fit"]),
        })

X = np.stack(windows)
window_info = pd.DataFrame(records)

train_mask = window_info["use_for_ae_fit"].to_numpy()
val_normal_mask = (
    window_info["subset"].eq("validation")
    & window_info["segment_anomaly"].eq(0)
).to_numpy()

X_train = X[train_mask]
X_val_normal = X[val_normal_mask]

assert np.isfinite(X).all()
assert X.shape[1:] == (WINDOW, 1)
assert len(X_train) > 0 and len(X_val_normal) > 0
assert len(window_info[["channel", "segment"]].drop_duplicates()) \
    == len(split_table)

np.savez_compressed(DATA_DIR / "windows_v1.npz", X=X)
window_info.to_csv(DATA_DIR / "window_index_v1.csv", index=False)

(SAVE_DIR / "models" / "window_config_v1.json").write_text(
    json.dumps({
        "window": WINDOW,
        "stride": STRIDE,
        "features": ["scaled_value"],
        "padding": False,
        "include_final_window": True,
    }, indent=2)
)

print("All windows:", X.shape)
print("Normal training windows:", X_train.shape)
print("Normal validation windows:", X_val_normal.shape)
print("\nWindow counts by subset:")
print(window_info["subset"].value_counts().to_string())
print("\nAll segments preserved. Windows and index saved.")

All windows: (74540, 8, 1)
Normal training windows: (29242, 8, 1)
Normal validation windows: (8385, 8, 1)

Window counts by subset:
subset
train         43153
test          19252
validation    12135

All segments preserved. Windows and index saved.


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from datetime import datetime, timezone

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
tf.keras.mixed_precision.set_global_policy("float32")

# Separate folder for each training attempt
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = SAVE_DIR / "models" / f"lstm_fp32_{run_id}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

inputs = tf.keras.Input(shape=(WINDOW, 1), name="telemetry")

encoded = tf.keras.layers.LSTM(
    16, activation="tanh", unroll=True, name="encoder"
)(inputs)

decoded = tf.keras.layers.RepeatVector(WINDOW)(encoded)

decoded = tf.keras.layers.LSTM(
    16,
    activation="tanh",
    return_sequences=True,
    unroll=True,
    name="decoder",
)(decoded)

outputs = tf.keras.layers.Dense(
    1, activation="linear", name="reconstruction"
)(decoded)

model = tf.keras.Model(inputs, outputs, name="satellite_lstm_ae")

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001, clipnorm=1.0
    ),
    loss="mse",
)

model.summary()

best_path = RUN_DIR / "best_model.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_path),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        mode="min",
    ),
    tf.keras.callbacks.CSVLogger(
        str(RUN_DIR / "training_history.csv")
    ),
    tf.keras.callbacks.TerminateOnNaN(),
]

history = model.fit(
    X_train,
    X_train,
    validation_data=(X_val_normal, X_val_normal),
    epochs=30,
    batch_size=128,
    shuffle=True,
    callbacks=callbacks,
    verbose=2,
)

losses = np.asarray(history.history["loss"])
val_losses = np.asarray(history.history["val_loss"])

assert np.isfinite(losses).all() and np.isfinite(val_losses).all(), \
    "Non-finite loss detected. Share the output before continuing."

# Reload the best saved checkpoint
model = tf.keras.models.load_model(best_path)

print("\nTraining complete.")
print("Best epoch:", int(np.argmin(val_losses)) + 1)
print("Best validation MSE:", float(val_losses.min()))
print("Model parameters:", model.count_params())
print("Saved model:", best_path)

Model: "satellite_lstm_ae"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ telemetry (InputLayer)          │ (None, 8, 1)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (LSTM)                  │ (None, 16)             │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 8, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (LSTM)                  │ (None, 8, 16)          │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction (Dense)          │ (None, 8, 1)           │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,281 (12.82 KB)

 Trainable params: 3,281 (12.82 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
229/229 - 7s - 33ms/step - loss: 0.1454 - val_loss: 0.0401
Epoch 2/30
229/229 - 2s - 9ms/step - loss: 0.0354 - val_loss: 0.0222
Epoch 3/30
229/229 - 2s - 7ms/step - loss: 0.0184 - val_loss: 0.0088
Epoch 4/30
229/229 - 2s - 8ms/step - loss: 0.0071 - val_loss: 0.0035
Epoch 5/30
229/229 - 2s - 8ms/step - loss: 0.0035 - val_loss: 0.0022
Epoch 6/30
229/229 - 3s - 11ms/step - loss: 0.0026 - val_loss: 0.0018
Epoch 7/30
229/229 - 3s - 11ms/step - loss: 0.0023 - val_loss: 0.0017
Epoch 8/30
229/229 - 2s - 7ms/step - loss: 0.0021 - val_loss: 0.0016
Epoch 9/30
229/229 - 2s - 7ms/step - loss: 0.0020 - val_loss: 0.0015
Epoch 10/30
229/229 - 2s - 8ms/step - loss: 0.0019 - val_loss: 0.0015
Epoch 11/30
229/229 - 2s - 7ms/step - loss: 0.0019 - val_loss: 0.0014
Epoch 12/30
229/229 - 2s - 8ms/step - loss: 0.0018 - val_loss: 0.0014
Epoch 13/30
229/229 - 2s - 8ms/step - loss: 0.0017 - val_loss: 0.0013
Epoch 14/30
229/229 - 3s - 13ms/step - loss: 0.0017 - val_loss: 0.0013
Epoch 15/30
229/229 - 4s 

In [ ]:
import json
import numpy as np
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

# Use validation windows only
val_mask = window_info["subset"].eq("validation").to_numpy()
X_validation = X[val_mask]

reconstructed = model.predict(
    X_validation, batch_size=256, verbose=0
)

window_mse = np.mean(
    (X_validation.astype(np.float64)
     - reconstructed.astype(np.float64)) ** 2,
    axis=(1, 2),
)
assert np.isfinite(window_mse).all()

val_windows = window_info.loc[val_mask].copy()
val_windows["window_mse"] = window_mse

# One score per labelled segment
val_scores = (
    val_windows.groupby(["channel", "segment"], as_index=False)
    .agg(
        anomaly=("segment_anomaly", "first"),
        score=("window_mse", "mean"),
        n_windows=("window_mse", "size"),
    )
)

assert len(val_scores) == 326

y_val = val_scores["anomaly"].to_numpy()
scores = val_scores["score"].to_numpy()

precision, recall, thresholds = precision_recall_curve(
    y_val, scores
)

denominator = precision[:-1] + recall[:-1]
candidate_f1 = np.divide(
    2 * precision[:-1] * recall[:-1],
    denominator,
    out=np.zeros_like(denominator),
    where=denominator > 0,
)

# For tied F1 values, choose the higher threshold
best_index = np.flatnonzero(
    candidate_f1 == candidate_f1.max()
)[-1]
THRESHOLD = float(thresholds[best_index])
predictions = (scores >= THRESHOLD).astype(int)

metrics = {
    "evaluation_set": "validation",
    "score": "mean_window_mse_per_segment",
    "threshold_selection": "maximum_validation_f1",
    "threshold": THRESHOLD,
    "precision": float(precision_score(
        y_val, predictions, zero_division=0)),
    "recall": float(recall_score(
        y_val, predictions, zero_division=0)),
    "f1": float(f1_score(
        y_val, predictions, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_val, scores)),
    "average_precision": float(
        average_precision_score(y_val, scores)),
}

val_scores["prediction"] = predictions
val_scores.to_csv(
    RUN_DIR / "validation_segment_scores.csv", index=False
)
(RUN_DIR / "validation_metrics.json").write_text(
    json.dumps(metrics, indent=2)
)

print(json.dumps(metrics, indent=2))
print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_val, predictions, labels=[0, 1]))
print("\nValidation results and threshold saved.")

{
  "evaluation_set": "validation",
  "score": "mean_window_mse_per_segment",
  "threshold_selection": "maximum_validation_f1",
  "threshold": 0.002197719445701078,
  "precision": 0.46236559139784944,
  "recall": 0.6323529411764706,
  "f1": 0.5341614906832298,
  "roc_auc": 0.7600889192886457,
  "average_precision": 0.5975541855242132
}

Confusion matrix [[TN, FP], [FN, TP]]:
[[208  50]
 [ 25  43]]

Validation results and threshold saved.


In [ ]:
from google.colab import drive
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf

drive.mount("/content/drive")

SAVE_DIR = Path(
    "/content/drive/MyDrive/Satellite_Anomaly_Detection_v1"
)
DATA_DIR = SAVE_DIR / "data" / "opssat_ad_v2"
SEED = 42

# Find completed runs with saved validation results
runs = sorted(
    p for p in (SAVE_DIR / "models").glob("lstm_fp32_*")
    if (p / "best_model.keras").exists()
    and (p / "validation_metrics.json").exists()
)

assert runs, "Saved model/results not found. Share this output."

# Match the baseline from our previous session
matches = []
for folder in runs:
    saved = json.loads(
        (folder / "validation_metrics.json").read_text()
    )
    if np.isclose(saved["f1"], 0.5341614906832298):
        matches.append(folder)

assert len(matches) == 1, (
    "Could not uniquely identify our baseline. Available runs:\n"
    + "\n".join(str(p) for p in runs)
)

RUN_DIR = matches[0]
model = tf.keras.models.load_model(
    RUN_DIR / "best_model.keras", compile=False
)

with np.load(DATA_DIR / "windows_v1.npz") as saved:
    X = saved["X"]

window_info = pd.read_csv(DATA_DIR / "window_index_v1.csv")
split_table = pd.read_csv(
    SAVE_DIR / "results" / "segment_split_v1.csv"
)
val_scores = pd.read_csv(
    RUN_DIR / "validation_segment_scores.csv"
)
metrics = json.loads(
    (RUN_DIR / "validation_metrics.json").read_text()
)

WINDOW = X.shape[1]
train_mask = (
    window_info["subset"].eq("train")
    & window_info["segment_anomaly"].eq(0)
).to_numpy()
val_mask = window_info["subset"].eq("validation").to_numpy()
val_normal_mask = (
    val_mask & window_info["segment_anomaly"].eq(0).to_numpy()
)

X_train = X[train_mask]
X_val_normal = X[val_normal_mask]
THRESHOLD = float(metrics["threshold"])

assert len(X) == len(window_info)
assert tuple(model.input_shape[1:]) == tuple(X.shape[1:])
assert np.isfinite(X).all()

print("Saved baseline restored.")
print("Run:", RUN_DIR.name)
print("Model parameters:", model.count_params())
print("Normal training windows:", X_train.shape)
print("Validation segments:", len(val_scores))
print("Previous validation F1:", metrics["f1"])
print("Previous validation ROC-AUC:", metrics["roc_auc"])
print("No training or test evaluation performed.")

Mounted at /content/drive
Saved baseline restored.
Run: lstm_fp32_20260929T185150250455Z
Model parameters: 3281
Normal training windows: (29242, 8, 1)
Validation segments: 326
Previous validation F1: 0.5341614906832298
Previous validation ROC-AUC: 0.7600889192886457
No training or test evaluation performed.


In [ ]:
from sklearn.metrics import (
    precision_recall_curve, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix,
)
import numpy as np
import pandas as pd

# 1. Channel-wise errors from the saved LSTM baseline
channel_rows = []

for channel, group in val_scores.groupby("channel"):
    y = group["anomaly"].to_numpy()
    pred = group["prediction"].to_numpy()

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    channel_rows.append({
        "channel": channel,
        "normal": int(tn + fp),
        "anomalies": int(tp + fn),
        "detected": int(tp),
        "missed": int(fn),
        "false_alarms": int(fp),
    })

channel_report = pd.DataFrame(channel_rows)
print("LSTM validation errors by channel:")
print(channel_report.to_string(index=False))

# 2. Simple baseline: squared distance from training mean
validation_mask = window_info["subset"].eq("validation").to_numpy()

simple_windows = window_info.loc[validation_mask].copy()
simple_windows["score"] = np.mean(
    X[validation_mask].astype(np.float64) ** 2,
    axis=(1, 2),
)

simple_scores = (
    simple_windows.groupby(["channel", "segment"], as_index=False)
    .agg(
        anomaly=("segment_anomaly", "first"),
        score=("score", "mean"),
    )
)

# Ensure both methods evaluate exactly the same segments
comparison = val_scores[
    ["channel", "segment", "anomaly", "score"]
].merge(
    simple_scores,
    on=["channel", "segment"],
    validate="one_to_one",
    suffixes=("_lstm", "_simple"),
)

assert len(comparison) == len(val_scores) == len(simple_scores)
assert comparison["anomaly_lstm"].eq(
    comparison["anomaly_simple"]
).all()

# 3. Tune each threshold using the same validation-F1 rule
def evaluate_validation(name, y, scores):
    assert np.isfinite(scores).all()
    p, r, thresholds = precision_recall_curve(y, scores)
    denominator = p[:-1] + r[:-1]
    f1_values = np.divide(
        2 * p[:-1] * r[:-1],
        denominator,
        out=np.zeros_like(denominator),
        where=denominator > 0,
    )

    best = np.flatnonzero(f1_values == f1_values.max())[-1]
    threshold = float(thresholds[best])
    pred = (scores >= threshold).astype(int)

    return {
        "method": name,
        "threshold": threshold,
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, scores),
        "average_precision": average_precision_score(y, scores),
    }

y = comparison["anomaly_lstm"].to_numpy()
report = pd.DataFrame([
    evaluate_validation(
        "LSTM reconstruction", y,
        comparison["score_lstm"].to_numpy()
    ),
    evaluate_validation(
        "Squared magnitude", y,
        comparison["score_simple"].to_numpy()
    ),
])

channel_report.to_csv(
    RUN_DIR / "validation_channel_errors.csv", index=False
)
report.to_csv(
    RUN_DIR / "validation_baseline_comparison.csv", index=False
)
simple_scores.to_csv(
    RUN_DIR / "validation_simple_scores.csv", index=False
)

print("\nValidation comparison — thresholds tuned on validation:")
print(report.round(4).to_string(index=False))
print("\nReports saved. Test set not evaluated.")

LSTM validation errors by channel:
 channel  normal  anomalies  detected  missed  false_alarms
CADC0872      63         20        15       5            21
CADC0873      74         15        12       3            12
CADC0874      20         10         6       4             0
CADC0884      25          0         0       0             7
CADC0886       1          1         1       0             0
CADC0888      28         10         6       4             6
CADC0890       1          2         2       0             0
CADC0892      27          6         0       6             1
CADC0894      19          4         1       3             3

Validation comparison — thresholds tuned on validation:
             method  threshold  precision  recall     f1  roc_auc  average_precision
LSTM reconstruction     0.0022     0.4624  0.6324 0.5342   0.7601             0.5976
  Squared magnitude     0.1328     0.2086  1.0000 0.3452   0.3332             0.1630

Reports saved. Test set not evaluated.


In [ ]:
# Step 12C: Compare mean vs maximum reconstruction error.
# Validation only; test set remains untouched.

validation_mask = window_info["subset"].eq("validation").to_numpy()
X_validation = X[validation_mask]

reconstructed = model.predict(
    X_validation, batch_size=256, verbose=0
)

window_errors = np.mean(
    (
        X_validation.astype(np.float64)
        - reconstructed.astype(np.float64)
    ) ** 2,
    axis=(1, 2),
)
assert np.isfinite(window_errors).all()

validation_windows = window_info.loc[
    validation_mask,
    ["channel", "segment", "segment_anomaly"]
].copy()
validation_windows["window_mse"] = window_errors

aggregation_scores = (
    validation_windows.groupby(["channel", "segment"], as_index=False)
    .agg(
        anomaly=("segment_anomaly", "first"),
        mean_score=("window_mse", "mean"),
        max_score=("window_mse", "max"),
    )
)

assert len(aggregation_scores) == 326

aggregation_report = pd.DataFrame([
    evaluate_validation(
        "LSTM mean error",
        aggregation_scores["anomaly"].to_numpy(),
        aggregation_scores["mean_score"].to_numpy(),
    ),
    evaluate_validation(
        "LSTM max error",
        aggregation_scores["anomaly"].to_numpy(),
        aggregation_scores["max_score"].to_numpy(),
    ),
])

aggregation_scores.to_csv(
    RUN_DIR / "validation_aggregation_scores.csv", index=False
)
aggregation_report.to_csv(
    RUN_DIR / "validation_aggregation_comparison.csv", index=False
)

print("Validation comparison — thresholds tuned on validation:")
print(aggregation_report.round(4).to_string(index=False))
print("\nSaved. Test set not evaluated.")

Validation comparison — thresholds tuned on validation:
         method  threshold  precision  recall     f1  roc_auc  average_precision
LSTM mean error     0.0022     0.4624  0.6324 0.5342   0.7601             0.5976
 LSTM max error     0.0216     0.9787  0.6765 0.8000   0.8865             0.8264

Saved. Test set not evaluated.


In [ ]:
# Step 12D: Save the selected FP32 scoring configuration.
import json
import numpy as np
import pandas as pd

comparison = pd.read_csv(
    RUN_DIR / "validation_aggregation_comparison.csv"
)
selected = comparison.loc[
    comparison["method"].eq("LSTM max error")
].iloc[0]

THRESHOLD = float(selected["threshold"])
assert np.isfinite(THRESHOLD)

scoring_config = {
    "model_file": "best_model.keras",
    "window_error": "mean_squared_error",
    "segment_aggregation": "max",
    "prediction_rule": "score >= threshold",
    "fp32_threshold": THRESHOLD,
    "selection_data": "validation",
    "threshold_selection": "maximum_validation_f1",
    "aggregation_candidates": ["mean", "max"],
    "validation_metrics": {
        name: float(selected[name])
        for name in [
            "precision", "recall", "f1",
            "roc_auc", "average_precision"
        ]
    },
    "test_evaluated": False,
}

config_path = RUN_DIR / "selected_scoring_config.json"
config_path.write_text(
    json.dumps(scoring_config, indent=2),
    encoding="utf-8"
)

print("Selected segment scoring: MAX window reconstruction MSE")
print("FP32 threshold:", repr(THRESHOLD))
print("Validation F1:", selected["f1"])
print("Saved:", config_path)
print("\nStep 12 complete. Next: Step 13 — FP32 TFLite conversion.")

Selected segment scoring: MAX window reconstruction MSE
FP32 threshold: 0.0216094808817206
Validation F1: 0.8
Saved: /content/drive/MyDrive/Satellite_Anomaly_Detection_v1/models/lstm_fp32_20260929T185150250455Z/selected_scoring_config.json

Step 12 complete. Next: Step 13 — FP32 TFLite conversion.


In [ ]:
# Step 13A: FP32 TFLite conversion + numerical sanity check
import tensorflow as tf
import numpy as np
import json

# Fixed batch size: one telemetry window per inference
@tf.function(input_signature=[
    tf.TensorSpec(
        shape=[1, int(X.shape[1]), int(X.shape[2])],
        dtype=tf.float32,
        name="telemetry"
    )
])
def serving_fn(inputs):
    return model(inputs, training=False)

concrete_fn = serving_fn.get_concrete_function()

converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [concrete_fn], model
)
converter.optimizations = []  # FP32; no quantization yet
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]

fp32_bytes = converter.convert()
fp32_path = RUN_DIR / "model_fp32.tflite"
fp32_path.write_bytes(fp32_bytes)

interpreter = tf.lite.Interpreter(
    model_content=fp32_bytes, num_threads=1
)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

assert input_info["dtype"] == np.float32
assert output_info["dtype"] == np.float32

# Reproducible validation sample; test data is not used
val_indices = np.flatnonzero(
    window_info["subset"].eq("validation").to_numpy()
)
rng = np.random.default_rng(42)
check_indices = rng.choice(
    val_indices, size=min(256, len(val_indices)), replace=False
)
check_x = X[check_indices].astype(np.float32)

keras_output = model.predict(
    check_x, batch_size=128, verbose=0
)
tflite_output = np.empty_like(keras_output)

for i in range(len(check_x)):
    interpreter.set_tensor(
        input_info["index"], check_x[i:i+1]
    )
    interpreter.invoke()
    tflite_output[i] = interpreter.get_tensor(
        output_info["index"]
    )[0]

assert np.isfinite(tflite_output).all()

difference = np.abs(
    keras_output.astype(np.float64)
    - tflite_output.astype(np.float64)
)
passed = bool(np.allclose(
    keras_output, tflite_output, rtol=1e-4, atol=1e-5
))

report = {
    "tensorflow_version": tf.__version__,
    "model_size_bytes": len(fp32_bytes),
    "validation_windows_checked": len(check_x),
    "max_absolute_difference": float(difference.max()),
    "mean_absolute_difference": float(difference.mean()),
    "rtol": 1e-4,
    "atol": 1e-5,
    "numerical_check_passed": passed,
}

(RUN_DIR / "fp32_conversion_check.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)
np.save(RUN_DIR / "fp32_check_window_indices.npy", check_indices)

print("Saved:", fp32_path)
print("FP32 TFLite size:", round(len(fp32_bytes) / 1024, 2), "KiB")
print("Input shape:", input_info["shape"])
print("Maximum output difference:", difference.max())
print("Mean output difference:", difference.mean())
print("Numerical check:", "PASS" if passed else "REVIEW NEEDED")
print("Test set not evaluated.")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


AssertionError: 

In [ ]:
# Step 13A diagnostic — no retraining or test-set evaluation
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Input finite:", np.isfinite(check_x).all())
print("Keras output finite:", np.isfinite(keras_output).all())
print("Input range:", float(check_x.min()), float(check_x.max()))
print("Previous TFLite NaNs:", int(np.isnan(tflite_output).sum()))
print("Previous TFLite Infs:", int(np.isinf(tflite_output).sum()))

resolver = tf.lite.experimental.OpResolverType

for name, mode in [
    ("Default", resolver.AUTO),
    ("Without default delegates",
     resolver.BUILTIN_WITHOUT_DEFAULT_DELEGATES),
    ("Reference kernels", resolver.BUILTIN_REF),
]:
    try:
        runner = tf.lite.Interpreter(
            model_content=fp32_bytes,
            num_threads=1,
            experimental_op_resolver_type=mode,
        )
        runner.allocate_tensors()
        inp = runner.get_input_details()[0]
        out = runner.get_output_details()[0]

        outputs = []
        for sample in check_x:
            runner.set_tensor(inp["index"], sample[None, ...])
            runner.invoke()
            outputs.append(runner.get_tensor(out["index"])[0])

        outputs = np.stack(outputs)
        finite = bool(np.isfinite(outputs).all())
        print(f"\n{name}:")
        print("  Output shape:", outputs.shape)
        print("  All outputs finite:", finite)

        if finite and np.isfinite(keras_output).all():
            diff = np.abs(
                outputs.astype(np.float64)
                - keras_output.astype(np.float64)
            )
            print("  Maximum difference:", float(diff.max()))
            print("  Matches Keras:", np.allclose(
                outputs, keras_output, rtol=1e-4, atol=1e-5
            ))
        else:
            print("  NaN count:", int(np.isnan(outputs).sum()))
            print("  Inf count:", int(np.isinf(outputs).sum()))

    except Exception as error:
        print(f"\n{name}: {type(error).__name__}: {error}")

print("\nDiagnostic complete. Test set not evaluated.")

TensorFlow: 2.20.0
Input finite: True
Keras output finite: True
Input range: -2.239410161972046 2.65476393699646
Previous TFLite NaNs: 2048
Previous TFLite Infs: 0

Default:
  Output shape: (256, 8, 1)
  All outputs finite: False
  NaN count: 2048
  Inf count: 0

Without default delegates: RuntimeError: tensorflow/lite/kernels/read_variable.cc:67 variable != nullptr was not true.Node number 8 (READ_VARIABLE) failed to invoke.

Reference kernels: RuntimeError: tensorflow/lite/kernels/read_variable.cc:67 variable != nullptr was not true.Node number 8 (READ_VARIABLE) failed to invoke.

Diagnostic complete. Test set not evaluated.


In [ ]:
# Step 13A repair: freeze weights before TFLite conversion
import tensorflow as tf
import numpy as np
import json
from tensorflow.python.framework.convert_to_constants import (
    convert_variables_to_constants_v2
)

# TensorFlow internal helper; record TF version for reproducibility.
frozen_fn = convert_variables_to_constants_v2(concrete_fn)

remaining_reads = [
    op.name for op in frozen_fn.graph.get_operations()
    if op.type == "ReadVariableOp"
]
assert not remaining_reads, remaining_reads
assert len(frozen_fn.inputs) == 1

# Verify freezing preserves the original output.
frozen_output = np.concatenate([
    frozen_fn(tf.convert_to_tensor(sample[None, ...]))[0].numpy()
    for sample in check_x
], axis=0)

assert np.isfinite(frozen_output).all()
np.testing.assert_allclose(
    frozen_output, keras_output, rtol=1e-4, atol=1e-5
)
print("Frozen TensorFlow model matches Keras.")

# Convert the frozen function without the original variable-bearing model.
converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [frozen_fn]
)
converter.optimizations = []
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]
repaired_bytes = converter.convert()

interpreter = tf.lite.Interpreter(
    model_content=repaired_bytes, num_threads=1
)
interpreter.allocate_tensors()
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

assert input_info["dtype"] == np.float32
assert output_info["dtype"] == np.float32

repaired_output = []
for sample in check_x:
    interpreter.set_tensor(input_info["index"], sample[None, ...])
    interpreter.invoke()
    repaired_output.append(
        interpreter.get_tensor(output_info["index"])[0]
    )

repaired_output = np.stack(repaired_output)
assert np.isfinite(repaired_output).all(), "Non-finite TFLite output"

difference = np.abs(
    repaired_output.astype(np.float64)
    - keras_output.astype(np.float64)
)
np.testing.assert_allclose(
    repaired_output, keras_output, rtol=1e-4, atol=1e-5
)

# Replace the earlier file only after successful verification.
fp32_bytes = repaired_bytes
tflite_output = repaired_output
fp32_path = RUN_DIR / "model_fp32.tflite"
fp32_path.write_bytes(fp32_bytes)

report = {
    "tensorflow_version": tf.__version__,
    "conversion": "frozen_concrete_function",
    "model_size_bytes": len(fp32_bytes),
    "validation_windows_checked": len(check_x),
    "max_absolute_difference": float(difference.max()),
    "mean_absolute_difference": float(difference.mean()),
    "rtol": 1e-4,
    "atol": 1e-5,
    "numerical_check_passed": True,
}
(RUN_DIR / "fp32_conversion_check.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)
np.save(RUN_DIR / "fp32_check_window_indices.npy", check_indices)

print("\nFP32 TFLite check: PASS")
print("Size:", round(len(fp32_bytes) / 1024, 2), "KiB")
print("Maximum difference:", difference.max())
print("Saved:", fp32_path)
print("Test set not evaluated.")

Frozen TensorFlow model matches Keras.

FP32 TFLite check: PASS
Size: 56.66 KiB
Maximum difference: 1.1920928955078125e-06
Saved: /content/drive/MyDrive/Satellite_Anomaly_Detection_v1/models/lstm_fp32_20260929T185150250455Z/model_fp32.tflite
Test set not evaluated.


In [ ]:
# Step 13B: Full validation comparison at the saved threshold
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

config = json.loads(
    (RUN_DIR / "selected_scoring_config.json").read_text()
)
threshold = float(config["fp32_threshold"])

mask = window_info["subset"].eq("validation").to_numpy()
validation_x = X[mask].astype(np.float32)

runner = tf.lite.Interpreter(
    model_path=str(RUN_DIR / "model_fp32.tflite"),
    num_threads=1
)
runner.allocate_tensors()
inp = runner.get_input_details()[0]
out = runner.get_output_details()[0]

errors = []
for sample in validation_x:
    runner.set_tensor(inp["index"], sample[None, ...])
    runner.invoke()
    reconstruction = runner.get_tensor(out["index"])[0]
    assert np.isfinite(reconstruction).all()
    errors.append(np.mean(
        (sample.astype(np.float64)
         - reconstruction.astype(np.float64)) ** 2
    ))

window_scores = window_info.loc[
    mask, ["channel", "segment", "segment_anomaly"]
].copy()
window_scores["error"] = errors

lite_scores = (
    window_scores.groupby(["channel", "segment"], as_index=False)
    .agg(
        anomaly=("segment_anomaly", "first"),
        tflite_score=("error", "max")
    )
)

keras_scores = pd.read_csv(
    RUN_DIR / "validation_aggregation_scores.csv",
    float_precision="round_trip"
)

comparison = keras_scores[
    ["channel", "segment", "anomaly", "max_score"]
].merge(
    lite_scores,
    on=["channel", "segment"],
    validate="one_to_one",
    suffixes=("_keras", "_tflite")
)
assert len(comparison) == len(keras_scores) == len(lite_scores) == 326
assert comparison["anomaly_keras"].equals(
    comparison["anomaly_tflite"]
)

y = comparison["anomaly_keras"].to_numpy()
rows = []

for name, column in [
    ("Keras FP32", "max_score"),
    ("TFLite FP32", "tflite_score")
]:
    scores = comparison[column].to_numpy()
    predictions = (scores >= threshold).astype(int)
    comparison[name + "_prediction"] = predictions

    rows.append({
        "model": name,
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, scores),
        "average_precision": average_precision_score(y, scores),
    })
    print(name, "confusion matrix:")
    print(confusion_matrix(y, predictions, labels=[0, 1]))

report = pd.DataFrame(rows)
disagreements = int((
    comparison["Keras FP32_prediction"]
    != comparison["TFLite FP32_prediction"]
).sum())

comparison.to_csv(
    RUN_DIR / "validation_fp32_tflite_scores.csv", index=False
)
report.to_csv(
    RUN_DIR / "validation_fp32_tflite_metrics.csv", index=False
)

print("\nSaved threshold:", repr(threshold))
print(report.round(6).to_string(index=False))
print("\nPrediction disagreements:", disagreements, "/ 326")
print("Test set not evaluated.")

Keras FP32 confusion matrix:
[[257   1]
 [ 22  46]]
TFLite FP32 confusion matrix:
[[257   1]
 [ 23  45]]

Saved threshold: 0.0216094808817206
      model  precision   recall       f1  roc_auc  average_precision
 Keras FP32   0.978723 0.676471 0.800000 0.886457           0.826434
TFLite FP32   0.978261 0.661765 0.789474 0.886457           0.826434

Prediction disagreements: 1 / 326
Test set not evaluated.


In [ ]:
# Inspect the single changed prediction.
changed = comparison.loc[
    comparison["Keras FP32_prediction"]
    != comparison["TFLite FP32_prediction"]
].copy()

changed["keras_margin"] = changed["max_score"] - threshold
changed["tflite_margin"] = changed["tflite_score"] - threshold
changed["score_difference"] = (
    changed["tflite_score"] - changed["max_score"]
)

columns = [
    "channel", "segment", "anomaly_keras",
    "max_score", "tflite_score",
    "keras_margin", "tflite_margin", "score_difference"
]

print("Threshold:", repr(threshold))
print(changed[columns].to_string(
    index=False,
    float_format=lambda value: f"{value:.17g}"
))

changed[columns].to_csv(
    RUN_DIR / "fp32_threshold_disagreement.csv", index=False
)

Threshold: 0.0216094808817206
 channel  segment  anomaly_keras            max_score         tflite_score           keras_margin           tflite_margin        score_difference
CADC0872      166              1 0.021609480881720634 0.021609456431671857 3.4694469519536142e-17 -2.4450048742574726e-08 -2.4450048777269195e-08


In [ ]:
# Step 14A: Reproducible calibration set from normal training only
import numpy as np
import pandas as pd
import json

rng = np.random.default_rng(42)

eligible = (
    window_info["subset"].eq("train")
    & window_info["segment_anomaly"].eq(0)
).to_numpy()

chosen = []

for channel in sorted(window_info.loc[eligible, "channel"].unique()):
    indices = np.flatnonzero(
        eligible & window_info["channel"].eq(channel).to_numpy()
    )

    # Random coverage plus the channel's training-range extremes.
    random_indices = rng.choice(
        indices, size=min(128, len(indices)), replace=False
    )
    channel_x = X[indices]
    low_index = indices[
        np.argmin(channel_x.min(axis=(1, 2)))
    ]
    high_index = indices[
        np.argmax(channel_x.max(axis=(1, 2)))
    ]

    chosen.extend(random_indices.tolist())
    chosen.extend([int(low_index), int(high_index)])

calibration_indices = np.unique(chosen)
calibration_x = X[calibration_indices].astype(np.float32)

assert eligible[calibration_indices].all()
assert np.isfinite(calibration_x).all()

calibration_manifest = window_info.iloc[calibration_indices].copy()
calibration_manifest.insert(0, "window_row", calibration_indices)

np.savez_compressed(
    RUN_DIR / "int8_calibration_v1.npz",
    X=calibration_x,
    window_indices=calibration_indices
)
calibration_manifest.to_csv(
    RUN_DIR / "int8_calibration_manifest.csv", index=False
)

calibration_config = {
    "seed": 42,
    "source": "normal_training_windows_only",
    "selection": "up_to_128_random_per_channel_plus_min_max_windows",
    "windows": len(calibration_indices),
    "input_min": float(calibration_x.min()),
    "input_max": float(calibration_x.max()),
}
(RUN_DIR / "int8_calibration_config.json").write_text(
    json.dumps(calibration_config, indent=2),
    encoding="utf-8"
)

print("Calibration shape:", calibration_x.shape)
print("\nWindows per channel:")
print(calibration_manifest.groupby("channel").size().to_string())
print("\nInput range:",
      calibration_config["input_min"],
      calibration_config["input_max"])
print("\nCalibration files saved. Validation/test not used.")

Calibration shape: (967, 8, 1)

Windows per channel:
channel
CADC0872    130
CADC0873    130
CADC0874    130
CADC0884    130
CADC0886     36
CADC0888    129
CADC0890     22
CADC0892    130
CADC0894    130

Input range: -2.4442663192749023 5.517440319061279

Calibration files saved. Validation/test not used.


In [ ]:
# Step 14B: INT8 conversion and basic verification
import tensorflow as tf
import numpy as np
import pandas as pd
import json

calibration_x = np.load(
    RUN_DIR / "int8_calibration_v1.npz"
)["X"].astype(np.float32)

def representative_dataset():
    for sample in calibration_x:
        yield [sample[None, ...]]

int8_converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [frozen_fn]
)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_dataset
int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

int8_bytes = int8_converter.convert()

int8_runner = tf.lite.Interpreter(
    model_content=int8_bytes, num_threads=1
)
int8_runner.allocate_tensors()

int8_input = int8_runner.get_input_details()[0]
int8_output = int8_runner.get_output_details()[0]

assert int8_input["dtype"] == np.int8
assert int8_output["dtype"] == np.int8

tensor_details = int8_runner.get_tensor_details()
floating_tensors = [
    t["name"] for t in tensor_details
    if np.issubdtype(t["dtype"], np.floating)
]
assert not floating_tensors, (
    f"Floating-point tensors remain: {floating_tensors[:10]}"
)

input_scale, input_zero = int8_input["quantization"]
output_scale, output_zero = int8_output["quantization"]
assert input_scale > 0 and output_scale > 0

# One inference to verify execution; accuracy is checked next.
sample = calibration_x[:1]
quantized_input = np.clip(
    np.rint(sample.astype(np.float64) / input_scale + input_zero),
    -128, 127
).astype(np.int8)

int8_runner.set_tensor(int8_input["index"], quantized_input)
int8_runner.invoke()
quantized_output = int8_runner.get_tensor(int8_output["index"])
decoded_output = (
    quantized_output.astype(np.float64) - output_zero
) * output_scale
assert np.isfinite(decoded_output).all()

int8_path = RUN_DIR / "model_int8.tflite"
int8_path.write_bytes(int8_bytes)

tensor_report = pd.DataFrame([
    {
        "index": t["index"],
        "name": t["name"],
        "dtype": np.dtype(t["dtype"]).name,
        "shape": str(t["shape"].tolist()),
    }
    for t in tensor_details
])
tensor_report.to_csv(
    RUN_DIR / "int8_tensor_audit.csv", index=False
)

fp32_size = (RUN_DIR / "model_fp32.tflite").stat().st_size
report = {
    "tensorflow_version": tf.__version__,
    "calibration_windows": len(calibration_x),
    "fp32_bytes": fp32_size,
    "int8_bytes": len(int8_bytes),
    "size_ratio_fp32_over_int8": fp32_size / len(int8_bytes),
    "input_scale": float(input_scale),
    "input_zero_point": int(input_zero),
    "output_scale": float(output_scale),
    "output_zero_point": int(output_zero),
    "floating_tensor_count": len(floating_tensors),
    "smoke_test_passed": True,
}
(RUN_DIR / "int8_conversion_report.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)

print("INT8 conversion and execution: PASS")
print("Input / output:", int8_input["dtype"], int8_output["dtype"])
print("Floating tensors:", len(floating_tensors))
print("Tensor types:", tensor_report["dtype"].value_counts().to_dict())
print("FP32 size:", round(fp32_size / 1024, 2), "KiB")
print("INT8 size:", round(len(int8_bytes) / 1024, 2), "KiB")
print("FP32 / INT8 size ratio:",
      round(fp32_size / len(int8_bytes), 3))
print("Saved:", int8_path)
print("\nDetection performance not yet checked. Test set untouched.")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 conversion and execution: PASS
Input / output: <class 'numpy.int8'> <class 'numpy.int8'>
Floating tensors: 0
Tensor types: {'int8': 305, 'int32': 10}
FP32 size: 56.66 KiB
INT8 size: 75.77 KiB
FP32 / INT8 size ratio: 0.748
Saved: /content/drive/MyDrive/Satellite_Anomaly_Detection_v1/models/lstm_fp32_20260929T185150250455Z/model_int8.tflite

Detection performance not yet checked. Test set untouched.


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
# Step 14C: INT8 validation at the unchanged FP32 threshold
import tensorflow as tf
import numpy as np
import pandas as pd
import json
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

config = json.loads(
    (RUN_DIR / "selected_scoring_config.json").read_text()
)
threshold = float(config["fp32_threshold"])

mask = window_info["subset"].eq("validation").to_numpy()
val_x = X[mask].astype(np.float32)

runner = tf.lite.Interpreter(
    model_path=str(RUN_DIR / "model_int8.tflite"),
    num_threads=1
)
runner.allocate_tensors()
inp = runner.get_input_details()[0]
out = runner.get_output_details()[0]

assert inp["dtype"] == out["dtype"] == np.int8
in_scale, in_zero = inp["quantization"]
out_scale, out_zero = out["quantization"]
assert in_scale > 0 and out_scale > 0

errors, clipped_counts = [], []

for sample in val_x:
    rounded = np.rint(
        sample.astype(np.float64) / in_scale + in_zero
    )
    clipped_counts.append(int(
        ((rounded < -128) | (rounded > 127)).sum()
    ))
    quantized = np.clip(rounded, -128, 127).astype(np.int8)

    runner.set_tensor(inp["index"], quantized[None, ...])
    runner.invoke()
    reconstruction = (
        runner.get_tensor(out["index"])[0].astype(np.float64)
        - out_zero
    ) * out_scale

    # Compare against original scaled input, before quantization/clipping.
    errors.append(np.mean(
        (sample.astype(np.float64) - reconstruction) ** 2
    ))

assert np.isfinite(errors).all()

window_results = window_info.loc[
    mask, ["channel", "segment", "segment_anomaly"]
].copy()
window_results["window_mse"] = errors
window_results["clipped_values"] = clipped_counts

int8_scores = (
    window_results.groupby(["channel", "segment"], as_index=False)
    .agg(
        anomaly=("segment_anomaly", "first"),
        score=("window_mse", "max"),
        clipped_values=("clipped_values", "sum"),
        n_windows=("window_mse", "size")
    )
)
assert len(int8_scores) == 326

y = int8_scores["anomaly"].to_numpy()
scores = int8_scores["score"].to_numpy()
predictions = (scores >= threshold).astype(int)
int8_scores["prediction"] = predictions

metrics = {
    "model": "TFLite INT8",
    "threshold": threshold,
    "precision": float(precision_score(y, predictions, zero_division=0)),
    "recall": float(recall_score(y, predictions, zero_division=0)),
    "f1": float(f1_score(y, predictions, zero_division=0)),
    "roc_auc": float(roc_auc_score(y, scores)),
    "average_precision": float(average_precision_score(y, scores)),
    "confusion_matrix": confusion_matrix(
        y, predictions, labels=[0, 1]
    ).tolist(),
    "clipped_window_value_fraction": float(
        np.sum(clipped_counts) / val_x.size
    ),
    "segments_with_clipping": int(
        (int8_scores["clipped_values"] > 0).sum()
    ),
}

int8_scores.to_csv(
    RUN_DIR / "validation_int8_segment_scores.csv", index=False
)
(RUN_DIR / "validation_int8_fixed_threshold_metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)

fp32_report = pd.read_csv(
    RUN_DIR / "validation_fp32_tflite_metrics.csv"
)
metric_columns = [
    "model", "precision", "recall", "f1",
    "roc_auc", "average_precision"
]
combined = pd.concat([
    fp32_report[metric_columns],
    pd.DataFrame([{k: metrics[k] for k in metric_columns}])
], ignore_index=True)
combined.to_csv(
    RUN_DIR / "validation_quantization_comparison.csv", index=False
)

print(combined.round(6).to_string(index=False))
print("\nINT8 confusion matrix:")
print(np.array(metrics["confusion_matrix"]))
print("\nClipped window values (%):",
      round(100 * metrics["clipped_window_value_fraction"], 4))
print("Segments with clipping:", metrics["segments_with_clipping"])
print("\nSaved. Threshold unchanged. Test set not evaluated.")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


      model  precision   recall       f1  roc_auc  average_precision
 Keras FP32   0.978723 0.676471 0.800000 0.886457           0.826434
TFLite FP32   0.978261 0.661765 0.789474 0.886457           0.826434
TFLite INT8   0.289340 0.838235 0.430189 0.820907           0.758450

INT8 confusion matrix:
[[118 140]
 [ 11  57]]

Clipped window values (%): 0.0
Segments with clipping: 0

Saved. Threshold unchanged. Test set not evaluated.


In [ ]:
# Step 14D: Separate validation-tuned thresholds for both TFLite models
import numpy as np
import pandas as pd
import json
from sklearn.metrics import (
    precision_recall_curve, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)

fp = pd.read_csv(
    RUN_DIR / "validation_fp32_tflite_scores.csv",
    float_precision="round_trip"
)
qi = pd.read_csv(
    RUN_DIR / "validation_int8_segment_scores.csv",
    float_precision="round_trip"
)

def tune_threshold(name, labels, scores):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    assert np.isfinite(scores).all()

    p, r, thresholds = precision_recall_curve(labels, scores)
    f1_values = np.divide(
        2 * p[:-1] * r[:-1],
        p[:-1] + r[:-1],
        out=np.zeros_like(thresholds),
        where=(p[:-1] + r[:-1]) > 0
    )

    # Same tie rule as before: highest threshold among equal best F1s
    best_index = np.flatnonzero(f1_values == f1_values.max())[-1]
    selected_threshold = float(thresholds[best_index])
    predictions = (scores >= selected_threshold).astype(int)
    tn, false_alarms, missed, detected = confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).ravel()

    return {
        "model": name,
        "threshold": selected_threshold,
        "precision": float(precision_score(
            labels, predictions, zero_division=0)),
        "recall": float(recall_score(
            labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(labels, scores)),
        "average_precision": float(
            average_precision_score(labels, scores)),
        "false_alarms": int(false_alarms),
        "missed": int(missed),
        "detected": int(detected),
    }

results = [
    tune_threshold(
        "TFLite FP32",
        fp["anomaly_keras"],
        fp["tflite_score"]
    ),
    tune_threshold(
        "TFLite INT8",
        qi["anomaly"],
        qi["score"]
    ),
]

report = pd.DataFrame(results)
report.to_csv(
    RUN_DIR / "validation_retuned_threshold_comparison.csv",
    index=False
)

(RUN_DIR / "tflite_validation_thresholds.json").write_text(
    json.dumps({
        "selection_data": "validation",
        "selection_rule": "maximum_f1_highest_threshold_on_ties",
        "segment_aggregation": "max_window_mse",
        "results": results,
        "test_evaluated": False
    }, indent=2),
    encoding="utf-8"
)

print("Validation results with separately tuned thresholds:")
print(report.round(6).to_string(index=False))
print("\nSaved separately. Earlier fixed-threshold results preserved.")
print("Test set not evaluated.")

Validation results with separately tuned thresholds:
      model  threshold  precision   recall       f1  roc_auc  average_precision  false_alarms  missed  detected
TFLite FP32   0.021609   0.978723 0.676471 0.800000 0.886457           0.826434             1      22        46
TFLite INT8   0.040433   0.741935 0.676471 0.707692 0.820907           0.758450            16      22        46

Saved separately. Earlier fixed-threshold results preserved.
Test set not evaluated.


In [ ]:
# Step 15A: Freeze this experiment before final test evaluation
import json
import hashlib
from datetime import datetime, timezone

threshold_data = json.loads(
    (RUN_DIR / "tflite_validation_thresholds.json").read_text()
)

model_files = {
    "TFLite FP32": "model_fp32.tflite",
    "TFLite INT8": "model_int8.tflite",
}

locked_models = []
for result in threshold_data["results"]:
    path = RUN_DIR / model_files[result["model"]]
    model_bytes = path.read_bytes()

    locked_models.append({
        "model": result["model"],
        "file": path.name,
        "sha256": hashlib.sha256(model_bytes).hexdigest(),
        "size_bytes": len(model_bytes),
        "threshold": result["threshold"],
    })

protocol = {
    "locked_at_utc": datetime.now(timezone.utc).isoformat(),
    "evaluation_unit": "segment",
    "window_error": "mse_against_original_scaled_input",
    "segment_aggregation": "max",
    "prediction_rule": "score >= threshold",
    "threshold_source": "validation_only_separately_per_model",
    "models": locked_models,
    "test_policy": "evaluate_both_models_without_further_test_based_tuning",
    "latency_plan": {
        "runtime": "tf.lite.Interpreter",
        "num_threads": 1,
        "batch_size": 1,
        "environment": "Colab CPU; not satellite hardware",
    },
}

lock_path = RUN_DIR / "evaluation_protocol_v1.json"

if lock_path.exists():
    existing = json.loads(lock_path.read_text())
    assert existing["models"] == locked_models, (
        "Existing lock differs. Do not overwrite."
    )
    print("Existing experiment lock verified.")
else:
    lock_path.write_text(
        json.dumps(protocol, indent=2), encoding="utf-8"
    )
    print("Experiment settings locked.")

for item in locked_models:
    print(item["model"], "| threshold:", repr(item["threshold"]))

print("Saved:", lock_path)
print("Ready for final test evaluation.")

Experiment settings locked.
TFLite FP32 | threshold: 0.021609456431671857
TFLite INT8 | threshold: 0.04043285887871799
Saved: /content/drive/MyDrive/Satellite_Anomaly_Detection_v1/models/lstm_fp32_20260929T185150250455Z/evaluation_protocol_v1.json
Ready for final test evaluation.


In [ ]:
# Step 15B: Final test evaluation using locked models and thresholds
import tensorflow as tf
import numpy as np
import pandas as pd
import json
import hashlib
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

protocol = json.loads(
    (RUN_DIR / "evaluation_protocol_v1.json").read_text()
)

test_mask = window_info["subset"].eq("test").to_numpy()
test_x = X[test_mask].astype(np.float32)
test_info = window_info.loc[
    test_mask, ["channel", "segment", "segment_anomaly"]
].copy()

assert np.isfinite(test_x).all()
assert len(test_info.drop_duplicates(["channel", "segment"])) == 529

# Verify both files before evaluating either model.
for item in protocol["models"]:
    data = (RUN_DIR / item["file"]).read_bytes()
    assert hashlib.sha256(data).hexdigest() == item["sha256"], (
        f"Model changed after locking: {item['model']}"
    )

test_reports = []

for item in protocol["models"]:
    print("Evaluating:", item["model"], flush=True)

    runner = tf.lite.Interpreter(
        model_path=str(RUN_DIR / item["file"]), num_threads=1
    )
    runner.allocate_tensors()
    inp = runner.get_input_details()[0]
    out = runner.get_output_details()[0]

    is_int8 = inp["dtype"] == np.int8
    if is_int8:
        assert out["dtype"] == np.int8
        in_scale, in_zero = inp["quantization"]
        out_scale, out_zero = out["quantization"]
        assert in_scale > 0 and out_scale > 0
    else:
        assert inp["dtype"] == out["dtype"] == np.float32

    errors, clipping = [], []

    for sample in test_x:
        if is_int8:
            rounded = np.rint(
                sample.astype(np.float64) / in_scale + in_zero
            )
            clipping.append(int(
                ((rounded < -128) | (rounded > 127)).sum()
            ))
            model_input = np.clip(
                rounded, -128, 127
            ).astype(np.int8)
        else:
            model_input = sample
            clipping.append(0)

        runner.set_tensor(inp["index"], model_input[None, ...])
        runner.invoke()
        reconstructed = runner.get_tensor(out["index"])[0]

        if is_int8:
            reconstructed = (
                reconstructed.astype(np.float64) - out_zero
            ) * out_scale
        else:
            reconstructed = reconstructed.astype(np.float64)

        assert np.isfinite(reconstructed).all()
        errors.append(np.mean(
            (sample.astype(np.float64) - reconstructed) ** 2
        ))

    windows = test_info.copy()
    windows["window_mse"] = errors
    windows["clipped_values"] = clipping

    segments = (
        windows.groupby(["channel", "segment"], as_index=False)
        .agg(
            anomaly=("segment_anomaly", "first"),
            score=("window_mse", "max"),
            n_windows=("window_mse", "size"),
            clipped_values=("clipped_values", "sum")
        )
    )
    assert len(segments) == 529
    assert np.isfinite(segments["score"]).all()

    y = segments["anomaly"].to_numpy()
    scores = segments["score"].to_numpy()
    threshold = float(item["threshold"])
    predictions = (scores >= threshold).astype(int)
    segments["prediction"] = predictions

    tn, fp, fn, tp = confusion_matrix(
        y, predictions, labels=[0, 1]
    ).ravel()

    test_reports.append({
        "model": item["model"],
        "threshold": threshold,
        "segments": len(segments),
        "precision": float(precision_score(
            y, predictions, zero_division=0)),
        "recall": float(recall_score(
            y, predictions, zero_division=0)),
        "f1": float(f1_score(y, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y, scores)),
        "average_precision": float(average_precision_score(y, scores)),
        "true_negatives": int(tn),
        "false_alarms": int(fp),
        "missed": int(fn),
        "detected": int(tp),
        "clipped_window_value_fraction": float(
            np.sum(clipping) / test_x.size
        ),
        "segments_with_clipping": int(
            (segments["clipped_values"] > 0).sum()
        ),
        "model_sha256": item["sha256"],
    })

    suffix = "int8" if is_int8 else "fp32"
    segments.to_csv(
        RUN_DIR / f"test_{suffix}_segment_scores.csv", index=False
    )

test_report = pd.DataFrame(test_reports)
test_report.to_csv(RUN_DIR / "final_test_metrics.csv", index=False)
(RUN_DIR / "final_test_metrics.json").write_text(
    json.dumps(test_reports, indent=2), encoding="utf-8"
)

columns = [
    "model", "precision", "recall", "f1", "roc_auc",
    "average_precision", "false_alarms", "missed", "detected"
]
print("\nFINAL TEST RESULTS:")
print(test_report[columns].round(6).to_string(index=False))
print("\nClipping:")
print(test_report[[
    "model", "clipped_window_value_fraction", "segments_with_clipping"
]].to_string(index=False))
print("\nResults saved. Locked thresholds unchanged.")

Evaluating: TFLite FP32


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluating: TFLite INT8


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



FINAL TEST RESULTS:
      model  precision   recall       f1  roc_auc  average_precision  false_alarms  missed  detected
TFLite FP32   0.919540 0.707965 0.800000 0.910632           0.871781             7      33        80
TFLite INT8   0.706422 0.681416 0.693694 0.863491           0.786450            32      36        77

Clipping:
      model  clipped_window_value_fraction  segments_with_clipping
TFLite FP32                       0.000000                       0
TFLite INT8                       0.000032                       3

Results saved. Locked thresholds unchanged.


In [ ]:
# Step 15C: Comparable FP32 / INT8 inference timing
import time
import platform
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

protocol = json.loads(
    (RUN_DIR / "evaluation_protocol_v1.json").read_text()
)

rng = np.random.default_rng(42)
val_indices = np.flatnonzero(
    window_info["subset"].eq("validation").to_numpy()
)
bench_indices = rng.choice(
    val_indices, size=min(512, len(val_indices)), replace=False
)
bench_x = X[bench_indices].astype(np.float32)

runners = {}
for item in protocol["models"]:
    data = (RUN_DIR / item["file"]).read_bytes()
    assert hashlib.sha256(data).hexdigest() == item["sha256"]

    runner = tf.lite.Interpreter(model_content=data, num_threads=1)
    runner.allocate_tensors()
    inp = runner.get_input_details()[0]
    out = runner.get_output_details()[0]

    # Prepare inputs outside timing: measures runtime inference only.
    if inp["dtype"] == np.int8:
        scale, zero = inp["quantization"]
        inputs = np.clip(
            np.rint(bench_x.astype(np.float64) / scale + zero),
            -128, 127
        ).astype(np.int8)
    else:
        inputs = bench_x

    runners[item["model"]] = (runner, inp, out, inputs)

    for i in range(100):
        runner.set_tensor(inp["index"], inputs[i % len(inputs):i % len(inputs)+1])
        runner.invoke()
        runner.get_tensor(out["index"])

records = []
names = list(runners)

# Alternate model order to reduce order-related timing bias.
for repeat in range(6):
    order = names if repeat % 2 == 0 else names[::-1]
    for name in order:
        runner, inp, out, inputs = runners[name]
        for i in range(len(inputs)):
            sample = inputs[i:i+1]
            start = time.perf_counter_ns()
            runner.set_tensor(inp["index"], sample)
            runner.invoke()
            runner.get_tensor(out["index"])
            elapsed_us = (time.perf_counter_ns() - start) / 1000
            records.append({
                "model": name,
                "repeat": repeat,
                "window_row": int(bench_indices[i]),
                "latency_us": elapsed_us
            })

raw = pd.DataFrame(records)
summary = (
    raw.groupby("model")["latency_us"]
    .agg(
        measurements="size",
        mean_us="mean",
        median_us="median",
        p95_us=lambda values: values.quantile(0.95)
    )
    .reset_index()
)

cpu_lines = [
    line.split(":", 1)[1].strip()
    for line in Path("/proc/cpuinfo").read_text().splitlines()
    if line.startswith("model name")
]
environment = {
    "tensorflow": tf.__version__,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cpu": cpu_lines[0] if cpu_lines else platform.processor(),
    "runtime": "tf.lite.Interpreter",
    "delegate_policy": "default",
    "num_threads": 1,
    "batch_size": 1,
    "warmup_calls_per_model": 100,
    "repeats": 6,
    "timed_operations": ["set_tensor", "invoke", "get_tensor"],
    "excluded": [
        "scaling", "window_creation", "input_quantization",
        "output_dequantization", "anomaly_scoring"
    ],
}

raw.to_csv(RUN_DIR / "latency_raw.csv", index=False)
summary.to_csv(RUN_DIR / "latency_summary.csv", index=False)
(RUN_DIR / "latency_environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

print("CPU:", environment["cpu"])
print("\nLatency per 8-reading window (microseconds):")
print(summary.round(3).to_string(index=False))

medians = summary.set_index("model")["median_us"]
ratio = medians["TFLite FP32"] / medians["TFLite INT8"]
print("\nFP32 / INT8 median latency ratio:", round(ratio, 3))
print("Ratio > 1: INT8 faster. Ratio < 1: INT8 slower.")
print("\nBenchmark saved.")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


CPU: Intel(R) Xeon(R) CPU @ 2.20GHz

Latency per 8-reading window (microseconds):
      model  measurements  mean_us  median_us  p95_us
TFLite FP32          3072   48.936     35.342  63.441
TFLite INT8          3072   49.067     33.690  59.162

FP32 / INT8 median latency ratio: 1.049
Ratio > 1: INT8 faster. Ratio < 1: INT8 slower.

Benchmark saved.
